# HyDE RAG v2: encode TREC-COVID on a free Colab GPU

Encodes the 171,332 TREC-COVID documents with **bge-base-en-v1.5** and **Contriever**, using
exactly the same settings as `src/hyde_rag/retrieval/dense.py`:
- same models and pooling (CLS + L2 normalisation for bge; mean pooling, unnormalised, for Contriever);
- max 512 tokens, documents formatted as `title + " " + text`, same document order;
- fp32 with TF32 off, for parity with the CPU runs.

**How to run**
1. *Runtime → Change runtime type → T4 GPU* (free).
2. *Runtime → Run all*. This takes about 20-40 minutes.
3. Two zip files download at the end (about 480 MB each). Extract both into the repo's
   `data\indexes\` folder. Each contains `trec-covid/<encoder>/embeddings.npy`, `doc_ids.json`
   and `meta.json`.
4. Back on the laptop, run `uv run hyde-rag verify-index --dataset trec-covid --encoder bge-base`
   (and `--encoder contriever`). This re-encodes a sample on CPU and checks it matches.

Costs nothing. If downloads are slow, set `USE_DRIVE = True` to save to Google Drive instead.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch the runtime to a GPU: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# Same transformers version as the project (pyproject.toml).
!pip -q install transformers==5.19.0

In [ ]:
import hashlib
import os
import urllib.request
import zipfile
from pathlib import Path

URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/trec-covid.zip"
MD5 = "ce62140cb23feb9becf6270d0d1fe6d1"  # as published in the BEIR README
if not os.path.exists("trec-covid.zip"):
    urllib.request.urlretrieve(URL, "trec-covid.zip")
digest = hashlib.md5(Path("trec-covid.zip").read_bytes()).hexdigest()
assert digest == MD5, f"md5 mismatch: {digest}"
with zipfile.ZipFile("trec-covid.zip") as zf:
    zf.extractall(".")
print("dataset ok")

In [ ]:
import json
import time

import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

torch.backends.cuda.matmul.allow_tf32 = False  # keep full fp32, as on CPU
torch.backends.cudnn.allow_tf32 = False

# Mirrors ENCODERS in src/hyde_rag/retrieval/dense.py
ENCODERS = {
    "bge-base": {
        "name": "bge-base",
        "model_id": "BAAI/bge-base-en-v1.5",
        "pooling": "cls",
        "normalize": True,
        "query_prefix": "Represent this sentence for searching relevant passages: ",
        "max_length": 512,
    },
    "contriever": {
        "name": "contriever",
        "model_id": "facebook/contriever",
        "pooling": "mean",
        "normalize": False,
        "query_prefix": "",
        "max_length": 512,
    },
}

# Same document order and text format as load_beir() + doc_text_dense().
doc_ids, texts = [], []
with open("trec-covid/corpus.jsonl", encoding="utf-8") as f:
    for line in f:
        o = json.loads(line)
        doc_ids.append(str(o["_id"]))
        title, text = o.get("title") or "", o.get("text") or ""
        texts.append(f"{title} {text}".strip() if title else text)
print(len(doc_ids), "documents")


def encode(spec, texts, batch_size=128):
    tok = AutoTokenizer.from_pretrained(spec["model_id"])
    model = AutoModel.from_pretrained(spec["model_id"]).cuda().eval()
    order = np.argsort([len(t) for t in texts])  # length-sorted batches, as in the project
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)
    with torch.inference_mode():
        for s in range(0, len(texts), batch_size):
            idx = order[s : s + batch_size]
            b = tok(
                [texts[i] for i in idx],
                padding=True,
                truncation=True,
                max_length=spec["max_length"],
                return_tensors="pt",
            ).to("cuda")
            h = model(**b).last_hidden_state
            if spec["pooling"] == "mean":
                m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
                e = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
            else:
                e = h[:, 0]
            if spec["normalize"]:
                e = torch.nn.functional.normalize(e, dim=-1)
            out[idx] = e.float().cpu().numpy()
            if (s // batch_size) % 200 == 0:
                print(f"  {spec['name']}: {s + len(idx):,}/{len(texts):,}", flush=True)
    return out

In [ ]:
for name, spec in ENCODERS.items():
    t0 = time.time()
    emb = encode(spec, texts)
    d = f"out/trec-covid/{name}"
    os.makedirs(d, exist_ok=True)
    np.save(f"{d}/embeddings.npy", emb)
    Path(f"{d}/doc_ids.json").write_text(json.dumps(doc_ids))
    meta = {**spec, "device": torch.cuda.get_device_name(0), "torch": torch.__version__}
    Path(f"{d}/meta.json").write_text(json.dumps(meta, indent=2))
    print(f"{name}: {emb.shape} in {(time.time() - t0) / 60:.1f} min")

In [ ]:
import shutil

USE_DRIVE = False  # True: copy to Google Drive instead of downloading through the browser

for name in ENCODERS:
    zip_path = shutil.make_archive(f"trec-covid-{name}", "zip", root_dir="out", base_dir=f"trec-covid/{name}")
    print(zip_path, round(os.path.getsize(zip_path) / 1e6), "MB")
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        os.makedirs("/content/drive/MyDrive/hyde-rag-v2", exist_ok=True)
        shutil.copy(zip_path, "/content/drive/MyDrive/hyde-rag-v2/")
    else:
        from google.colab import files

        files.download(zip_path)